# 🎯 Customer Churn Prediction Model Training
## Mamina Baby Spa

Notebook ini digunakan untuk:
1. Load data dari PostgreSQL
2. Exploratory Data Analysis (EDA)
3. Feature Engineering & Selection
4. Model Training (XGBoost)
5. Model Evaluation
6. SHAP Explainability
7. Export Model untuk Production

---

## 1. Setup & Import Libraries

In [ ]:
# Install dependencies jika belum ada
# !pip install pandas numpy scikit-learn xgboost shap sqlalchemy psycopg2-binary matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Database
from sqlalchemy import create_engine

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

# Explainability
import shap

# Model Persistence
import joblib
import pickle

print("✅ Libraries imported successfully!")
print(f"XGBoost version: {xgb.__version__}")
print(f"SHAP version: {shap.__version__}")

## 2. Database Connection

In [ ]:
# Konfigurasi Database - sesuaikan dengan .env backend
DB_CONFIG = {
    'host': 'localhost',
    'port': 5433,
    'database': 'maminaChurn_db',
    'user': 'postgres',
    'password': 'admin'  # Ganti dengan password Anda
}

DATABASE_URL = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

engine = create_engine(DATABASE_URL)

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute("SELECT 1")
        print("✅ Database connected successfully!")
except Exception as e:
    print(f"❌ Database connection failed: {e}")

## 3. Load Data dari Database

In [ ]:
# Load Customer Features
query_features = """
SELECT 
    cf.customer_id,
    cf.r_score,
    cf.f_score,
    cf.m_score,
    cf.tenure_days,
    cf.avg_sentiment_30,
    cf.neg_msg_count_30,
    cf.avg_response_secs,
    cf.intensity_7d,
    cf.updated_at
FROM customer_features cf
WHERE cf.r_score IS NOT NULL
"""

df_features = pd.read_sql(query_features, engine)
print(f"✅ Loaded {len(df_features)} customer features")
df_features.head()

In [ ]:
# Load Customer Info
query_customers = """
SELECT 
    customer_id,
    name,
    is_active,
    first_visit,
    last_visit,
    total_visits,
    total_spending
FROM customers
"""

df_customers = pd.read_sql(query_customers, engine)
print(f"✅ Loaded {len(df_customers)} customers")
df_customers.head()

In [ ]:
# Merge data
df = df_features.merge(df_customers, on='customer_id', how='left')
print(f"✅ Merged dataset: {len(df)} rows")
df.info()

## 4. Define Churn Label

**Definisi Churn:**
- Customer dianggap churn jika tidak berkunjung dalam 90 hari terakhir
- Atau `is_active = False`

In [ ]:
# Definisi churn
CHURN_THRESHOLD_DAYS = 90
today = datetime.now()

def define_churn(row):
    """Define churn based on last visit and is_active status"""
    if row['is_active'] == False:
        return 1  # Churned
    
    if pd.isna(row['last_visit']):
        return 1  # No visit = churned
    
    days_since_last_visit = (today - pd.to_datetime(row['last_visit'])).days
    
    if days_since_last_visit > CHURN_THRESHOLD_DAYS:
        return 1  # Churned
    
    return 0  # Active

df['churn'] = df.apply(define_churn, axis=1)

print("\n📊 Churn Distribution:")
print(df['churn'].value_counts())
print(f"\nChurn Rate: {df['churn'].mean()*100:.2f}%")

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview
print("📊 Dataset Overview:")
print(f"Total samples: {len(df)}")
print(f"Features: {df.shape[1]}")
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Churn distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
labels = ['Active', 'Churned']
sizes = df['churn'].value_counts().sort_index()
axes[0].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Churn Distribution')

# Bar chart
sns.countplot(data=df, x='churn', palette=colors, ax=axes[1])
axes[1].set_xticklabels(labels)
axes[1].set_title('Churn Count')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by churn status
feature_cols = ['r_score', 'f_score', 'm_score', 'tenure_days', 
                'avg_sentiment_30', 'neg_msg_count_30', 'avg_response_secs', 'intensity_7d']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    if col in df.columns:
        sns.boxplot(data=df, x='churn', y=col, palette=colors, ax=axes[i])
        axes[i].set_xticklabels(['Active', 'Churned'])
        axes[i].set_title(f'{col} by Churn Status')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation_cols = feature_cols + ['churn']
corr_matrix = df[correlation_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 6. Data Preprocessing

In [ ]:
# Select features for model
FEATURE_COLUMNS = [
    'r_score',           # Recency score
    'f_score',           # Frequency score
    'm_score',           # Monetary score
    'tenure_days',       # Customer tenure
    'avg_sentiment_30',  # Average sentiment last 30 days
    'neg_msg_count_30',  # Negative message count last 30 days
    'avg_response_secs', # Average response time
    'intensity_7d'       # Message intensity last 7 days
]

# Prepare X and y
X = df[FEATURE_COLUMNS].copy()
y = df['churn'].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Handle missing values
print("Missing values before:")
print(X.isnull().sum())

# Fill missing values with median
for col in X.columns:
    if X[col].isnull().sum() > 0:
        X[col].fillna(X[col].median(), inplace=True)

print("\nMissing values after:")
print(X.isnull().sum())

In [ ]:
# Train-Test Split (stratified untuk handle imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining churn rate: {y_train.mean()*100:.2f}%")
print(f"Test churn rate: {y_test.mean()*100:.2f}%")

In [ ]:
# Feature Scaling (untuk beberapa model)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled successfully!")

## 7. Model Training - XGBoost

In [ ]:
# Calculate class weights for imbalanced data
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
scale_pos_weight = class_weights[1] / class_weights[0]
print(f"Scale pos weight: {scale_pos_weight:.2f}")

In [ ]:
# XGBoost Model dengan hyperparameters yang sudah di-tune
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,  # Handle imbalanced
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# Train model
print("Training XGBoost model...")
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

print("✅ Model training completed!")

In [ ]:
# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')

print(f"\n📊 Cross-Validation Results (5-Fold):")
print(f"ROC-AUC scores: {cv_scores}")
print(f"Mean ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

## 8. Model Evaluation

In [ ]:
# Predictions
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Metrics
print("📊 Model Evaluation on Test Set:")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred_proba):.4f}")

In [ ]:
# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

In [ ]:
# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Active', 'Churned'],
            yticklabels=['Active', 'Churned'])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis')
plt.title('Feature Importance (XGBoost)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print("\n📊 Feature Importance:")
print(feature_importance)

## 9. SHAP Explainability

In [ ]:
# Create SHAP explainer
print("Creating SHAP explainer...")
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values untuk test set
shap_values = explainer.shap_values(X_test)

print("✅ SHAP explainer created!")

In [ ]:
# SHAP Summary Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLUMNS, show=False)
plt.title('SHAP Summary Plot')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP Bar Plot (Mean absolute SHAP values)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLUMNS, plot_type='bar', show=False)
plt.title('Mean |SHAP Value| - Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Individual prediction explanation (sample)
sample_idx = 0  # Index customer yang ingin dijelaskan

print(f"\n🔍 Explanation for Customer at index {sample_idx}:")
print(f"Actual: {'Churned' if y_test.iloc[sample_idx] == 1 else 'Active'}")
print(f"Predicted probability of churn: {y_pred_proba[sample_idx]:.4f}")

# Force plot
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[sample_idx], X_test.iloc[sample_idx], 
               feature_names=FEATURE_COLUMNS, matplotlib=True)

## 10. Export Model untuk Production

Export model dan SHAP explainer ke folder `backend/models/` agar bisa digunakan oleh Flask backend.

In [ ]:
import os

# Path ke folder backend/models
MODEL_DIR = '../backend/models'
os.makedirs(MODEL_DIR, exist_ok=True)

# Paths
MODEL_PATH = os.path.join(MODEL_DIR, 'churn_model.pkl')
EXPLAINER_PATH = os.path.join(MODEL_DIR, 'shap_explainer.pkl')
SCALER_PATH = os.path.join(MODEL_DIR, 'scaler.pkl')
METADATA_PATH = os.path.join(MODEL_DIR, 'model_metadata.pkl')

print(f"Model directory: {os.path.abspath(MODEL_DIR)}")

In [ ]:
# Save XGBoost Model
print("Saving XGBoost model...")
joblib.dump(xgb_model, MODEL_PATH)
print(f"✅ Model saved to: {MODEL_PATH}")

# Save SHAP Explainer
print("\nSaving SHAP explainer...")
joblib.dump(explainer, EXPLAINER_PATH)
print(f"✅ Explainer saved to: {EXPLAINER_PATH}")

# Save Scaler
print("\nSaving scaler...")
joblib.dump(scaler, SCALER_PATH)
print(f"✅ Scaler saved to: {SCALER_PATH}")

In [ ]:
# Save Model Metadata
metadata = {
    'feature_columns': FEATURE_COLUMNS,
    'training_date': datetime.now().isoformat(),
    'n_samples': len(df),
    'churn_rate': float(df['churn'].mean()),
    'model_type': 'XGBClassifier',
    'metrics': {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred)),
        'recall': float(recall_score(y_test, y_pred)),
        'f1_score': float(f1_score(y_test, y_pred)),
        'roc_auc': float(roc_auc_score(y_test, y_pred_proba))
    },
    'hyperparameters': {
        'n_estimators': 100,
        'max_depth': 5,
        'learning_rate': 0.1,
        'subsample': 0.8,
        'colsample_bytree': 0.8
    }
}

joblib.dump(metadata, METADATA_PATH)
print(f"\n✅ Metadata saved to: {METADATA_PATH}")
print("\nMetadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

In [ ]:
# Verify saved files
print("\n📁 Saved Files:")
for filename in os.listdir(MODEL_DIR):
    filepath = os.path.join(MODEL_DIR, filename)
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"  {filename}: {size_mb:.2f} MB")

## 11. Test Load Model (Simulasi Production)

In [ ]:
# Simulasi loading model seperti di Flask
print("Loading model untuk testing...")

loaded_model = joblib.load(MODEL_PATH)
loaded_explainer = joblib.load(EXPLAINER_PATH)
loaded_metadata = joblib.load(METADATA_PATH)

print("✅ Model loaded successfully!")
print(f"Model type: {type(loaded_model).__name__}")
print(f"Feature columns: {loaded_metadata['feature_columns']}")

In [ ]:
# Test prediction dengan sample data
sample_features = X_test.iloc[0:5]
sample_predictions = loaded_model.predict_proba(sample_features)[:, 1]

print("\n🧪 Sample Predictions:")
for i, (idx, row) in enumerate(sample_features.iterrows()):
    actual = 'Churned' if y_test.iloc[i] == 1 else 'Active'
    prob = sample_predictions[i]
    predicted = 'Churned' if prob > 0.5 else 'Active'
    print(f"  Sample {i+1}: Actual={actual}, Predicted={predicted}, Prob={prob:.4f}")

In [ ]:
# Test SHAP explanation
sample_shap = loaded_explainer.shap_values(sample_features.iloc[0:1])

print("\n🔍 SHAP Explanation for Sample 1:")
for feat, shap_val in zip(FEATURE_COLUMNS, sample_shap[0]):
    direction = "↑ churn" if shap_val > 0 else "↓ churn"
    print(f"  {feat}: {shap_val:+.4f} ({direction})")

---

## ✅ Training Complete!

Model telah disimpan di folder `backend/models/`:

| File | Deskripsi |
|------|-----------|
| `churn_model.pkl` | XGBoost model untuk prediksi |
| `shap_explainer.pkl` | SHAP explainer untuk interpretability |
| `scaler.pkl` | StandardScaler untuk feature scaling |
| `model_metadata.pkl` | Metadata training (metrics, hyperparameters) |

**Next Steps:**
1. Jalankan Flask backend
2. Test endpoint `/api/v1/predictions/` untuk melihat prediksi
3. Test endpoint `/api/v1/predictions/<customer_id>/explain` untuk SHAP explanation